# Centre update rule on the room dataset -- nine runs

Three arms across three seeds:

| arm | what it is |
|---|---|
| `none` | cross-entropy only. **Control**: at seed 42 it should land close to the existing baseline run (validation 0.5683, test 0.5633). A consistency check, not a replay — the image order differs by design and the historical run used a different GPU. |
| `gradient` | the Centre Loss variant used throughout this thesis |
| `wen` | the original centre-update rule, Equation (4) and Algorithm 1 of Wen et al. |

Nine runs at roughly two hours each. **Colab will disconnect before this
finishes.** That is expected and handled: every completed arm is written to
Drive, and re-running the training cell skips arms that are already finished
and verified. Just reconnect and run the cells again from the top.

Run the cells in order. Nothing here writes to the folders of any other
experiment.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Code -- pinned to one commit, so nine runs across several sessions all use
#    the same code.
#
# PINNED_COMMIT is the commit these runs are defined against. Leaving it at None
# would follow main, and if main moved between two Colab sessions the nine arms
# would be produced by different code while the configuration looked identical.
# Set it to None deliberately only if you mean to follow the branch.
#
# If the repository is private this also needs a token: create a fine-grained
# personal access token on GitHub with read-only Contents permission for this
# repository, then add it in Colab under the key icon as GH_TOKEN and enable it
# for this notebook. If the repository is public no secret is needed.
import os
import subprocess

REPO_DIR = '/content/room-classification'
REPO = 'SilviuBR24/room-classification'
BRANCH = 'main'
PINNED_COMMIT = '093cf7379b076b7f866f9365f9fc5021ae16c2df'
# This is the commit that carries the training code these nine runs
# are defined against. A commit cannot contain its own hash, so the
# pin necessarily names the commit before this one; that commit holds
# every file under rooms_wen/, and the commit adding this line changes
# nothing but this notebook.

PUBLIC_URL = 'https://github.com/' + REPO + '.git'

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GH_TOKEN')
except Exception:
    TOKEN = None

if TOKEN:
    AUTH_URL = 'https://x-access-token:' + TOKEN + '@github.com/' + REPO + '.git'
    print('using the GH_TOKEN secret')
else:
    AUTH_URL = PUBLIC_URL
    print('no GH_TOKEN secret found; cloning without authentication')


def git(*args, cwd=None):
    """Run git, keeping the token out of anything that reaches the screen."""
    r = subprocess.run(['git'] + list(args), cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout + r.stderr
    if TOKEN:
        out = out.replace(TOKEN, '***')
    if r.returncode != 0:
        shown = ' '.join('<url>' if (TOKEN and TOKEN in a) else a for a in args)
        raise SystemExit('git ' + shown + ' failed:' + chr(10) + out)
    return out


if not os.path.exists(REPO_DIR):
    git('clone', AUTH_URL, REPO_DIR)

# Hard reset rather than pull. A reused Colab runtime keeps the previous
# session's clone, and `git pull` can leave it behind without reporting a
# failure -- which once caused a whole run to be silently skipped and reported
# as a success.
git('remote', 'set-url', 'origin', AUTH_URL, cwd=REPO_DIR)
git('fetch', '--all', '--prune', '--tags', cwd=REPO_DIR)
if PINNED_COMMIT:
    git('checkout', '--detach', PINNED_COMMIT, cwd=REPO_DIR)
    git('reset', '--hard', PINNED_COMMIT, cwd=REPO_DIR)
else:
    git('checkout', '-B', BRANCH, 'origin/' + BRANCH, cwd=REPO_DIR)
    git('reset', '--hard', 'origin/' + BRANCH, cwd=REPO_DIR)
git('clean', '-fd', cwd=REPO_DIR)

# The token would otherwise stay in .git/config for the life of the runtime.
# Nothing below needs to reach GitHub again, so put the plain URL back.
git('remote', 'set-url', 'origin', PUBLIC_URL, cwd=REPO_DIR)

head = git('rev-parse', 'HEAD', cwd=REPO_DIR).strip()
print('HEAD is now:', head)
print('one-line:   ', git('log', '--oneline', '-1', cwd=REPO_DIR).strip())
if PINNED_COMMIT and head != PINNED_COMMIT:
    raise SystemExit('HEAD is ' + head + ' but the pin asks for ' + PINNED_COMMIT)
dirty = git('status', '--porcelain', cwd=REPO_DIR).strip()
if dirty:
    raise SystemExit('the working tree is not clean after reset:' + chr(10) + dirty)
print('working tree clean, pinned as intended')

!pip install -q -r {REPO_DIR}/vit_s16_baseline/requirements.txt

%cd {REPO_DIR}
print('rooms_wen present:', os.path.isdir(REPO_DIR + '/rooms_wen'))


In [ ]:
# 4. Dataset -- the same split the 3200 runs used
import os, time
SRC = '/content/drive/MyDrive/Dissertation_Thesis/dataset_split.zip'
DST = '/content/dataset_split'

if os.path.isdir(DST):
    print('already extracted')
else:
    t0 = time.time()
    !unzip -q {SRC} -d /content/
    print(f'extracted in {time.time()-t0:.0f}s')

In [ ]:
# 5. Verify the split with the loader that will actually read it
#
# Counting *.jpg with a recursive glob would not describe what training sees.
# RoomDataset accepts five extensions, and it reads only files sitting directly
# in each class directory. A .png would be loaded but not counted, and an image
# in a subdirectory would be counted but never loaded. So the check builds the
# dataset itself, and then separately looks for the two kinds of file the
# dataset would treat differently from a glob.
import sys
from pathlib import Path

import yaml

sys.path.insert(0, '/content/room-classification/rooms_wen')
from rooms_wen_data import IMAGE_SUFFIXES, RoomDataset

CFG = yaml.safe_load(
    open('/content/room-classification/rooms_wen/config_rooms_wen.yaml',
         encoding='utf-8'))
CLASSES = list(CFG['data']['class_names'])
ROOT = Path('/content/dataset_split')
EXPECTED_PER_CLASS = {'train': 3200, 'val': 300, 'eval': 100}

problems = []

for part, per_class in EXPECTED_PER_CLASS.items():
    d = ROOT / part
    if not d.is_dir():
        problems.append(f'{part}: directory missing')
        continue

    # What the dataset would actually load.
    try:
        ds = RoomDataset(d, CLASSES)
    except SystemExit as exc:
        problems.append(f'{part}: RoomDataset refused it -- {exc}')
        continue

    counts = ds.class_counts()
    wrong = {c: n for c, n in counts.items() if n != per_class}
    if wrong:
        problems.append(f'{part}: wrong counts {wrong}, expected {per_class} each')
    if ds.class_to_idx != {c: i for i, c in enumerate(CLASSES)}:
        problems.append(f'{part}: class mapping is {ds.class_to_idx}')

    # Directories that are not one of the six classes.
    extra_dirs = sorted(x.name for x in d.iterdir()
                        if x.is_dir() and x.name not in CLASSES)
    if extra_dirs:
        problems.append(f'{part}: unexpected class directories {extra_dirs}')

    # Files the dataset would load but a *.jpg count would miss, and files
    # nested one level deeper, which the dataset would never see.
    non_jpg = nested = 0
    for c in CLASSES:
        cd = d / c
        for f in cd.iterdir():
            if f.is_dir():
                nested += sum(1 for x in f.rglob('*')
                              if x.suffix.lower() in IMAGE_SUFFIXES)
            elif f.suffix.lower() in IMAGE_SUFFIXES and f.suffix.lower() != '.jpg':
                non_jpg += 1
    if non_jpg:
        problems.append(f'{part}: {non_jpg} loadable non-.jpg images '
                        f'(a *.jpg count would have missed these)')
    if nested:
        problems.append(f'{part}: {nested} images in subdirectories, which '
                        f'RoomDataset does not read')

    total = len(ds)
    print(f'{"OK " if not wrong else "BAD"} {part:6s} {total:6d} images, '
          f'{per_class} per class, mapping {list(ds.class_to_idx)[:2]}...')

if problems:
    for p in problems:
        print('PROBLEM:', p)
    raise SystemExit('the split does not match the protocol; stop here')

print()
print('Split verified with RoomDataset itself: 3200 train, 300 val, '
      '100 test per class.')
print('This checks the files on this runtime. It says nothing about the')
print('integrity of dataset_split.zip on Drive, which was not opened here.')


In [ ]:
# 6. Train and evaluate the nine arms.
#
# Safe to re-run after a disconnect: finished arms are skipped, and an arm is
# only reused if its saved configuration matches the current one.
#
# --only lets you do it in pieces, e.g. --only none  then  --only gradient
!cd /content/room-classification/rooms_wen && python compare_rooms_wen.py     --runs-dir /content/drive/MyDrive/Dissertation_Thesis/dissertation_runs

In [ ]:
# 7. Where things stand
import glob, os, pandas as pd
RUNS = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
csv = os.path.join(RUNS, 'rooms_wen_comparison_results.csv')
if os.path.isfile(csv):
    df = pd.read_csv(csv)
    print(df[['center_mode','seed','best_val_accuracy','test_accuracy','minutes']]
          .to_string(index=False))
    done = df['test_accuracy'].notna().sum()
    print()
    print(f'{done}/9 arms finished')
else:
    print('no summary yet -- the first arm has not finished')

print()
print('run directories on Drive:')
for d in sorted(glob.glob(os.path.join(RUNS, '*rooms_wen_*'))):
    print('  ', os.path.basename(d))

In [ ]:
# 8. Did the control reproduce the existing baseline?
#
# The seed-42 `none` arm and 2026-07-11_17-49_vit_s16_3200_baseline are the same
# configuration trained by two different loops. If they disagree, this folder's
# loop differs somewhere and the comparison is not yet trustworthy.
import glob, os, re

RUNS = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
def acc(pattern):
    hits = sorted(glob.glob(os.path.join(RUNS, pattern)))
    if not hits: return None, None
    ev = sorted(glob.glob(os.path.join(hits[-1], 'outputs', 'eval_*')))
    if not ev: return hits[-1], None
    txt = open(os.path.join(ev[-1], 'metrics.txt'), encoding='utf-8').read()
    m = re.search(r'Overall accuracy:\s*([0-9.]+)', txt)
    return hits[-1], (float(m.group(1)) if m else None)

_, a = acc('*_vit_s16_3200_baseline')
_, b = acc('*_rooms_wen_none_s42')
print(f'existing baseline (shared loop): {a}')
print(f'control arm  (this loop, s42)  : {b}')
if a is not None and b is not None:
    print(f'difference: {(b-a)*100:+.2f}pp')
    print('The two loops consume randomness differently, so an exact match is not')
    print('expected; a difference of a few points would need explaining.')